# BERTopic: Temporal Topic Analysis (Native)

Uses BERTopic's native `topics_over_time()` with evolution & global tuning.
Topic -1 (outliers) excluded.

> **Note**: BERTopic hardcodes `[:5]` words in `topics_over_time()`. We patch it to `[:10]` at runtime.

In [1]:
import time
import importlib
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from bertopic import BERTopic
import bertopic._bertopic as _bmod
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/bertopic/tuning/hdbscan")
RESULT_DIR = Path("../../../../results/bertopic/temporal")
VERSION = "v1"
TOP_N_WORDS = 10
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

In [3]:
def rbo(list_1, list_2, p=0.9):
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    score = 0.0
    for d in range(1, k + 1):
        agreement = len(set(list_1[:d]) & set(list_2[:d])) / d
        score += (p ** (d - 1)) * agreement
    return score * (1 - p)


def calculate_irbo(topics_words_list, p=0.9):
    if len(topics_words_list) < 2:
        return 0.0
    scores = [1.0 - rbo(topics_words_list[i], topics_words_list[j], p)
              for i, j in combinations(range(len(topics_words_list)), 2)]
    return np.mean(scores)

## Patch BERTopic: `[:5]` → `[:10]`

BERTopic's `topics_over_time()` hardcodes `[:5]` when building the Words column (line 937).
We patch the source file at runtime to use `[:10]` instead, then reload the module.

In [4]:
# Patch BERTopic source: [:5] → [:TOP_N_WORDS]
_BERTOPIC_SOURCE_PATH = _bmod.__file__

with open(_BERTOPIC_SOURCE_PATH) as f:
    _ORIGINAL_SOURCE = f.read()

# Count occurrences to verify
_old = '[words[0] for words in values][:5]'
_new = f'[words[0] for words in values][:{TOP_N_WORDS}]'
n_matches = _ORIGINAL_SOURCE.count(_old)
print(f"Found {n_matches} occurrences of '[:5]' pattern in {_BERTOPIC_SOURCE_PATH}")

_PATCHED_SOURCE = _ORIGINAL_SOURCE.replace(_old, _new)

with open(_BERTOPIC_SOURCE_PATH, 'w') as f:
    f.write(_PATCHED_SOURCE)

# Reload module and patch the existing BERTopic class
importlib.reload(_bmod)
BERTopic.topics_over_time = _bmod.BERTopic.topics_over_time

print(f"✅ Patched: topics_over_time() now outputs {TOP_N_WORDS} words per topic")
print(f"⚠️  Run the 'Restore' cell at the bottom when done!")

Found 0 occurrences of '[:5]' pattern in /home/nedo/Kuliah/TA/Program/.venv/lib/python3.13/site-packages/bertopic/_bertopic.py
✅ Patched: topics_over_time() now outputs 10 words per topic
⚠️  Run the 'Restore' cell at the bottom when done!


## Load Models & Data + topics_over_time

In [5]:
all_models = {}
all_data = {}
all_years = {}
all_tot = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")

    model = BERTopic.load(str(MODEL_DIR / f"best_{subject}_quality"))
    all_models[subject] = model

    df = pd.read_csv(BASE_DIR / subject / "emb" / f"{VERSION}.csv")
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    df["topic"] = model.topics_

    # Native topics_over_time (now patched to output TOP_N_WORDS)
    print(f"  Computing topics_over_time...")
    start = time.time()
    tot_df = model.topics_over_time(
        docs=df["text"].tolist(),
        timestamps=df["year"].tolist(),
        evolution_tuning=True, global_tuning=True,
    )
    elapsed = time.time() - start
    all_tot[subject] = tot_df

    n_outliers = (df["topic"] == -1).sum()
    df = df[df["topic"] != -1].reset_index(drop=True)
    all_data[subject] = df
    years = sorted(df["year"].unique())
    all_years[subject] = years

    n_topics = len(set(model.topics_)) - (1 if -1 in model.topics_ else 0)
    # Verify word count
    sample_words = tot_df.iloc[0]["Words"].split(", ")
    print(f"  {subject}: {len(df):,} docs (excl. {n_outliers:,} outliers), "
          f"{n_topics} topics, {len(years)} years, {len(sample_words)} words/topic [{elapsed:.1f}s]")

print(f"\n✅ All subjects loaded")


Loading cs...
  Computing topics_over_time...
  cs: 98,534 docs (excl. 67,222 outliers), 261 topics, 26 years, 10 words/topic [45.9s]

Loading math...
  Computing topics_over_time...
  math: 87,138 docs (excl. 69,947 outliers), 150 topics, 26 years, 10 words/topic [28.6s]

Loading physics...
  Computing topics_over_time...
  physics: 80,033 docs (excl. 66,278 outliers), 232 topics, 26 years, 10 words/topic [47.4s]

✅ All subjects loaded


## Topic Prevalence Over Time

In [6]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    all_topic_ids = sorted(df["topic"].unique())

    global_tw = {}
    for tid in all_topic_ids:
        info = model.get_topic(tid)
        global_tw[tid] = ", ".join([w for w, _ in info[:5]]) if info else ""

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        counts = year_df["topic"].value_counts()
        for tid in all_topic_ids:
            count = counts.get(tid, 0)
            rows.append({"subject": subject, "year": year, "topic_id": tid,
                         "doc_count": count, "total_docs_year": len(year_df),
                         "proportion": round(count / len(year_df), 6) if len(year_df) > 0 else 0,
                         "top_words": global_tw.get(tid, "")})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_prevalence.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_prevalence.csv")

  CS: Saved topic_prevalence.csv
  MATH: Saved topic_prevalence.csv
  PHYSICS: Saved topic_prevalence.csv


## Topic Word Evolution (Native BERTopic)

In [7]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    tot_df = all_tot[subject]
    tot_filtered = tot_df[tot_df["Topic"] != -1].copy()

    topic_words_per_year = {}
    rows = []
    for _, row in tot_filtered.iterrows():
        year = int(row["Timestamp"].year) if hasattr(row["Timestamp"], "year") else int(row["Timestamp"])
        tid = int(row["Topic"])
        words_str = row["Words"]
        if isinstance(words_str, str):
            words = [w.strip() for w in words_str.split(", ")][:TOP_N_WORDS]
        else:
            words = list(words_str)[:TOP_N_WORDS]

        topic_words_per_year[(year, tid)] = words
        rows.append({"subject": subject, "year": year,
                     "topic_id": tid, "top_words": ", ".join(words),
                     "frequency": int(row["Frequency"])})

    all_topic_words_per_year[subject] = topic_words_per_year
    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_word_evolution.csv", index=False)

    # Verify word count
    sample = rows[0]["top_words"].split(", ") if rows else []
    print(f"  {subject.upper()}: Saved topic_word_evolution.csv ({len(rows)} rows, {len(sample)} words/topic)")

  CS: Saved topic_word_evolution.csv (4328 rows, 10 words/topic)
  MATH: Saved topic_word_evolution.csv (3572 rows, 10 words/topic)
  PHYSICS: Saved topic_word_evolution.csv (5162 rows, 10 words/topic)


## Per-Year Coherence & IRBO

In [8]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    topic_words_per_year = all_topic_words_per_year[subject]
    n_topics = len(set(model.topics_)) - (1 if -1 in model.topics_ else 0)

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        year_texts = [t.split() for t in year_df["text"].tolist()]
        dictionary = Dictionary(year_texts)

        active_topics = sorted(year_df["topic"].unique())
        year_tw = [topic_words_per_year[(year, t)] for t in active_topics
                   if (year, t) in topic_words_per_year
                   and len(topic_words_per_year[(year, t)]) >= 2]

        if year_tw:
            cm = CoherenceModel(topics=year_tw, texts=year_texts,
                                dictionary=dictionary, coherence='c_v', processes=1)
            coherence = cm.get_coherence()
        else:
            coherence = 0.0

        irbo = calculate_irbo(year_tw, p=RBO_P)
        quality = 2 * coherence * irbo / (coherence + irbo) if (coherence + irbo) > 0 else 0.0

        print(f"  {year}: {len(year_df):>6,} docs | {len(active_topics):>3} topics | "
              f"Q={quality:.4f} (C={coherence:.4f}, IRBO={irbo:.4f})")

        rows.append({"subject": subject, "year": year, "num_docs": len(year_df),
                     "num_topics_total": n_topics, "num_topics_active": len(active_topics),
                     "coherence_cv": round(coherence, 6), "irbo_mean": round(irbo, 6),
                     "topic_quality": round(quality, 6)})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "per_year_metrics.csv", index=False)
    print(f"  Saved: per_year_metrics.csv")


Per-Year Metrics: CS (261 topics)
  2000:    211 docs |  47 topics | Q=0.7795 (C=0.6391, IRBO=0.9988)
  2001:    224 docs |  63 topics | Q=0.8076 (C=0.6776, IRBO=0.9993)
  2002:    306 docs |  65 topics | Q=0.8104 (C=0.6816, IRBO=0.9991)
  2003:    384 docs |  74 topics | Q=0.8072 (C=0.6770, IRBO=0.9994)
  2004:    436 docs |  95 topics | Q=0.7711 (C=0.6277, IRBO=0.9993)
  2005:    495 docs |  87 topics | Q=0.7866 (C=0.6486, IRBO=0.9991)
  2006:    501 docs |  88 topics | Q=0.7840 (C=0.6451, IRBO=0.9989)
  2007:    503 docs |  82 topics | Q=0.7503 (C=0.6007, IRBO=0.9991)
  2008:    508 docs | 100 topics | Q=0.7721 (C=0.6292, IRBO=0.9991)
  2009:    528 docs | 108 topics | Q=0.7675 (C=0.6231, IRBO=0.9990)
  2010:    703 docs | 126 topics | Q=0.7324 (C=0.5780, IRBO=0.9992)
  2011:    854 docs | 144 topics | Q=0.7336 (C=0.5797, IRBO=0.9991)
  2012:  1,217 docs | 161 topics | Q=0.7264 (C=0.5706, IRBO=0.9991)
  2013:  1,413 docs | 178 topics | Q=0.6941 (C=0.5319, IRBO=0.9990)
  2014:  1,56

## Topic Trends

In [9]:
from scipy.stats import linregress

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    # Get topic words from model
    global_tw = {}
    for tid in [t for t in df['topic'].unique() if t != -1]:
        info = model.get_topic(tid)
        if info:
            global_tw[tid] = [w for w, _ in info[:5]]
        else:
            global_tw[tid] = ['?']

    rows = []
    topic_ids = sorted([t for t in df["topic"].unique() if t != -1])
    for tid in topic_ids:
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        # Align proportions with all years
        all_years = sorted(total_per_year.index)
        prop_aligned = proportions.reindex(all_years, fill_value=0.0)

        years_arr = np.array(all_years, dtype=float)
        props_arr = prop_aligned.values.astype(float)

        # Linear regression
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, props_arr)

        # Early/late for display
        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean() if len(topic_years) >= 5 else proportions.mean()
        late_mean = proportions[topic_years[-5:]].mean() if len(topic_years) >= 5 else proportions.mean()

        # Classify by slope significance
        if p_val < 0.05 and slope > 0:
            trend_label = "GROWING"
        elif p_val < 0.05 and slope < 0:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        top_words = global_tw.get(tid, ["?"])
        rows.append({
            "subject": subject, "topic_id": tid,
            "top_words": ", ".join(top_words),
            "total_docs": len(topic_df),
            "first_year": year_counts.index.min(),
            "last_year": year_counts.index.max(),
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "slope": round(slope, 8),
            "r_squared": round(r_val**2, 4),
            "p_value": round(p_val, 6),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")


  CS: Growing=160, Stable=72, Declining=29
  MATH: Growing=59, Stable=49, Declining=42
  PHYSICS: Growing=98, Stable=83, Declining=51


## Top 5 Growing & Declining Topics

In [10]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df['trend'] == 'GROWING'].sort_values('slope', ascending=False)
    declining = trends_df[trends_df['trend'] == 'DECLINING'].sort_values('slope', ascending=True)

    print(f"\n  " + chr(0x1F4C8) + " TOP 5 GROWING (steepest positive slope):")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | {row['top_words']}")

    print(f"\n  " + chr(0x1F4C9) + " TOP 5 DECLINING (steepest negative slope):")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | {row['top_words']}")



  CS

  📈 TOP 5 GROWING (steepest positive slope):
    T  2 | slope=+0.001113 R²=0.690 | 0.0031 → 0.0237 | visual, multimodal, vision, image, language
    T  7 | slope=+0.000825 R²=0.789 | 0.0028 → 0.0175 | graph, node, gnns, graphs, gnn
    T  5 | slope=+0.000823 R²=0.854 | 0.0038 → 0.0171 | recommendation, recommender, item, user, items
    T  8 | slope=+0.000750 R²=0.609 | 0.0050 → 0.0144 | adversarial, attacks, robustness, attack, perturbations
    T  3 | slope=+0.000716 R²=0.508 | 0.0083 → 0.0189 | policy, reinforcement, rl, reward, learning

  📉 TOP 5 DECLINING (steepest negative slope):
    T  4 | slope=-0.007493 R²=0.607 | 0.1780 → 0.0083 | logic, calculus, type, semantics, programs
    T  0 | slope=-0.004151 R²=0.676 | 0.1089 → 0.0248 | graphs, graph, log, vertex, gg
    T  1 | slope=-0.003745 R²=0.610 | 0.1050 → 0.0222 | quantum, classical, circuits, entanglement, qubit
    T131 | slope=-0.002059 R²=0.206 | 0.0574 → 0.0010 | grid, hpc, computing, workflows, workflow
    T 25

## Evolution Summary

In [11]:
for subject in LIST_SUBJECT:
    metrics_df = pd.read_csv(RESULT_DIR / subject / "per_year_metrics.csv")
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")
    n_topics = len(set(all_models[subject].topics_)) - (1 if -1 in all_models[subject].topics_ else 0)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])

    summary = {"subject": subject, "num_topics": n_topics,
               "num_years": len(metrics_df),
               "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
               "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
               "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
               "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
               "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
               "quality_std": round(metrics_df["topic_quality"].std(), 6),
               "topics_growing": g, "topics_stable": s, "topics_declining": d}

    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / "evolution_summary.csv", index=False)
    print(f"  {subject.upper()}: C={summary['coherence_mean']:.4f}, "
          f"IRBO={summary['irbo_mean']:.4f}, Q={summary['quality_mean']:.4f} | "
          f"↑{g} →{s} ↓{d}")

  CS: C=0.5926, IRBO=0.9987, Q=0.7422 | ↑160 →72 ↓29
  MATH: C=0.5137, IRBO=0.9979, Q=0.6773 | ↑59 →49 ↓42
  PHYSICS: C=0.5632, IRBO=0.9987, Q=0.7198 | ↑98 →83 ↓51


## Restore BERTopic Source

**Always run this cell** after completing the analysis to restore the original `[:5]` in `_bertopic.py`.

In [12]:
with open(_BERTOPIC_SOURCE_PATH, 'w') as f:
    f.write(_ORIGINAL_SOURCE)

importlib.reload(_bmod)
BERTopic.topics_over_time = _bmod.BERTopic.topics_over_time

print(f"✅ Restored original BERTopic source: [:5]")

✅ Restored original BERTopic source: [:5]
